# Data coverage and source checks

This notebook checks the supplied tables and documents their coverage. Dataset descriptions are in `FEATURES_CATALOG.md`; the research task is in `TRIAL_README.md`. The target is the Johnson Matthey New York rhodium series from 16 September 1996 onward. Missing observations are distinguished from absent reporting periods and unavailable early history.

## Data
### Load the supplied snapshot

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent
files = sorted((root / "data").glob("bds_*.parquet"))
assert len(files) == 8, "Review catalog scope if the input files have changed."
frames = {p.stem.split("__", 1)[1]: pd.read_parquet(p) for p in files}
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 12)
manifest = pd.DataFrame([
    {"dataset": p.stem.split("__", 1)[1], "file": p.name,
     "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in files
])
display(manifest)

,dataset,file,sha256
0,chinadata_ev_production,bds_06c4bc4f03a645bf951a92f8e59c3006__chinadat...,114ceddac9ff2c3dc5409197d17ed8761bd5fe21fafbf6...
1,pgm_price_history,bds_5529cd6bc41841d7b58fb69e90cd2546__pgm_pric...,a3febcdf24cdf160c388ba54695d039fa8a9b1bad06458...
2,chinadata_vehicle_production,bds_59f19703af7c45bab999c0ed86851136__chinadat...,36c40bdac6f97dea5d038981b6ea4158474d4eae655c17...
3,fred_zafprmito01ixobm_history,bds_68ec2edb578d4f93af8bc60fc0db9777__fred_zaf...,51daa4c11a4272e701c7b8a9954fabefeebedc3d995038...
4,iea_evs_eu_cars_historical,bds_8a05db43d9ba4fe58ec076250a56f1f9__iea_evs_...,77f9c0276a7f536a4f56d8bac38e6031123c511faa5b78...
5,fred_altsales_history,bds_92934197439c46fc9622dfc440cace8f__fred_alt...,babe33bbd9f82b1c0dbf38f3e3fa384b34d6b1bbbc7d6f...
6,un_comtrade_sa_refined_rhodium_monthly,bds_fb39b19cbcc4464abc5ca4dbbdf4cf1f__un_comtr...,7208d4bbac14304cc026ef9883b833bf04568014388d53...
7,fred_dexsfus_history,bds_fed88706ea3046d4a66c7c7c7c71ef12__fred_dex...,6d2e52634b693a5e757302a84b8411f1d5a0d1edac63fd...


## Results
### File completeness and validity

In [2]:
inventory = []
field_issues = []
for name, frame in frames.items():
    numeric = frame.select_dtypes("number")
    inventory.append({"dataset": name, "rows": len(frame), "columns": len(frame.columns),
                      "duplicate_rows": int(frame.duplicated().sum()),
                      "null_cells": int(frame.isna().sum().sum()),
                      "infinite_values": int(np.isinf(numeric).sum().sum())})
    for column in frame:
        values = frame[column]
        null_count = int(values.isna().sum())
        sentinel_count = int(values.astype("string").str.strip().str.lower().isin(["", "n/a", "na", "null", "unknown", "."]).sum())
        if null_count or sentinel_count:
            field_issues.append({"dataset": name, "column": column,
                                 "nulls": null_count, "null_pct": round(100 * null_count / len(frame), 2),
                                 "string_sentinels": sentinel_count})
display(pd.DataFrame(inventory))
display(pd.DataFrame(field_issues))

,dataset,rows,columns,duplicate_rows,null_cells,infinite_values
0,chinadata_ev_production,35,7,0,0,0
1,pgm_price_history,15875,10,0,0,0
2,chinadata_vehicle_production,323,7,0,0,0
3,fred_zafprmito01ixobm_history,586,9,0,0,0
4,iea_evs_eu_cars_historical,157,8,0,0,0
5,fred_altsales_history,608,9,0,0,0
6,un_comtrade_sa_refined_rhodium_monthly,196,16,0,195,0
7,fred_dexsfus_history,12183,9,0,474,0


,dataset,column,nulls,null_pct,string_sentinels
0,un_comtrade_sa_refined_rhodium_monthly,net_weight_kg,1,0.51,0
1,un_comtrade_sa_refined_rhodium_monthly,quantity_unit,0,0.00,3
2,un_comtrade_sa_refined_rhodium_monthly,alternate_quantity_kg,194,98.98,0
3,fred_dexsfus_history,value,474,3.89,0


### Temporal coverage and logical keys
Missing grid periods are absent rows, separate from null values on existing rows.

In [3]:
coverage = []
gaps = {}
for name, frame in frames.items():
    if "year" in frame:
        groups = frame.groupby(["parameter", "powertrain", "unit"])
        logical_key = ["region", "category", "parameter", "mode", "powertrain", "year", "unit"]
    else:
        groups = frame.groupby("metal_code") if "metal_code" in frame else [("all", frame)]
        logical_key = ["metal_code", "observation_date"] if "metal_code" in frame else (["period"] if "period" in frame else ["series_id", "observation_date"])
    assert not frame.duplicated(logical_key).any(), (name, "duplicate key")
    for key, group in groups:
        label = " / ".join(key) if isinstance(key, tuple) else key
        if "year" in group:
            observed = sorted(group.year.unique())
            missing = sorted(set(range(min(observed), max(observed) + 1)) - set(observed))
            first, last, grid = str(min(observed)), str(max(observed)), "annual"
        else:
            column = "period" if "period" in group else "observation_date"
            fmt = "%Y%m" if "comtrade" in name else None
            dates = pd.to_datetime(group[column], format=fmt, errors="raise")
            assert dates.notna().all()
            observed = pd.DatetimeIndex(dates.unique()).sort_values()
            monthly = column == "period" or ("frequency" in group and group.frequency.iloc[0].lower() == "monthly")
            frequency = "MS" if monthly else ("D" if "pgm" in name else "B")
            missing = pd.date_range(observed.min(), observed.max(), freq=frequency).difference(observed).strftime("%Y-%m-%d").tolist()
            first, last, grid = str(observed.min().date()), str(observed.max().date()), frequency
        coverage.append({"dataset": name, "series": label, "rows": len(group),
                         "first": first, "last": last, "grid": grid, "absent_periods": len(missing)})
        if missing:
            gaps[f"{name}: {label}"] = missing
coverage = pd.DataFrame(coverage)
display(coverage)
for series, missing in gaps.items():
    print(series + ": " + ", ".join(missing))

,dataset,series,rows,first,last,grid,absent_periods
0,chinadata_ev_production,all,35,2023-03-01,2026-07-01,MS,6
1,pgm_price_history,ir,3175,2018-01-01,2026-09-10,D,0
2,pgm_price_history,pd,3175,2018-01-01,2026-09-10,D,0
3,pgm_price_history,pt,3175,2018-01-01,2026-09-10,D,0
4,pgm_price_history,rh,3175,2018-01-01,2026-09-10,D,0
5,pgm_price_history,ru,3175,2018-01-01,2026-09-10,D,0
6,chinadata_vehicle_production,all,323,1995-02-01,2026-07-01,MS,55
7,fred_zafprmito01ixobm_history,all,586,1975-01-01,2023-10-01,MS,0
8,iea_evs_eu_cars_historical,EV sales / BEV / Vehicles,16,2010,2025,annual,0
9,iea_evs_eu_cars_historical,EV sales / EV / Vehicles,16,2010,2025,annual,0


chinadata_ev_production: all: 2024-01-01, 2024-02-01, 2025-01-01, 2025-02-01, 2026-01-01, 2026-02-01
chinadata_vehicle_production: all: 1996-01-01, 1996-02-01, 1996-03-01, 1996-04-01, 1996-05-01, 1996-06-01, 1996-07-01, 1996-08-01, 1996-09-01, 1996-10-01, 1996-11-01, 1996-12-01, 1997-01-01, 1998-01-01, 1999-01-01, 2000-01-01, 2001-01-01, 2002-01-01, 2003-01-01, 2004-01-01, 2005-01-01, 2006-01-01, 2007-01-01, 2008-01-01, 2009-01-01, 2010-01-01, 2011-01-01, 2012-01-01, 2013-01-01, 2013-02-01, 2014-01-01, 2014-02-01, 2015-01-01, 2016-01-01, 2016-02-01, 2017-01-01, 2017-02-01, 2018-01-01, 2018-02-01, 2019-01-01, 2019-02-01, 2020-01-01, 2020-02-01, 2021-01-01, 2021-02-01, 2022-01-01, 2022-02-01, 2023-01-01, 2023-02-01, 2024-01-01, 2024-02-01, 2025-01-01, 2025-02-01, 2026-01-01, 2026-02-01
un_comtrade_sa_refined_rhodium_monthly: all: 2020-12-01


### PGM price-field consistency
Rows on weekends and apparent OHLC inconsistencies warrant checking the source conventions. They are not proof of bad prices.

In [4]:
pgm = frames["pgm_price_history"]
checks = []
for metal, group in pgm.groupby("metal_code"):
    group = group.sort_values("observation_date")
    dates = pd.to_datetime(group.observation_date)
    checks.append({"metal": metal, "weekend_rows": int((dates.dt.dayofweek >= 5).sum()),
        "weekend_price_changes": int(((dates.dt.dayofweek >= 5) & group.price_usd.diff().ne(0)).sum()),
        "price_outside_low_high": int(((group.price_usd < group.low_price_usd) | (group.price_usd > group.high_price_usd)).sum()),
        "open_outside_low_high": int(((group.opening_price_usd < group.low_price_usd) | (group.opening_price_usd > group.high_price_usd)).sum()),
        "close_outside_low_high": int(((group.closing_price_usd < group.low_price_usd) | (group.closing_price_usd > group.high_price_usd)).sum()),
        "price_differs_from_close": int(group.price_usd.ne(group.closing_price_usd).sum()),
        "reversed_range": int(group.low_price_usd.gt(group.high_price_usd).sum()),
        "identical_ohlc": int(group[["opening_price_usd", "high_price_usd", "low_price_usd", "closing_price_usd"]].nunique(axis=1).eq(1).sum())})
display(pd.DataFrame(checks))

,metal,weekend_rows,weekend_price_changes,price_outside_low_high,open_outside_low_high,close_outside_low_high,price_differs_from_close,reversed_range,identical_ohlc
0,ir,906,0,0,0,0,0,0,3175
1,pd,906,800,23,71,0,994,0,437
2,pt,906,848,17,72,0,994,0,274
3,rh,906,0,0,0,0,0,0,3175
4,ru,906,0,0,0,0,0,0,3175


### Trade quantity and FX missingness

In [5]:
trade = frames["un_comtrade_sa_refined_rhodium_monthly"]
trade_issues = trade.net_weight_kg.isna() | trade.net_weight_kg.le(0) | trade.quantity_unit.ne("kg") | trade.unit_value_usd_per_kg.isna()
display(trade.loc[trade_issues, ["period", "primary_value_usd", "net_weight_kg", "quantity", "quantity_unit", "unit_value_usd_per_kg"]])
valid_weight = trade.net_weight_kg.gt(0)
print("Maximum unit-value reconciliation error:",
      (trade.loc[valid_weight, "primary_value_usd"] / trade.loc[valid_weight, "net_weight_kg"] - trade.loc[valid_weight, "unit_value_usd_per_kg"]).abs().max())
fx = frames["fred_dexsfus_history"]
print("FX missing values:", int(fx.value.isna().sum()), "of", len(fx),
      "({:.2f}%)".format(100 * fx.value.isna().mean()))
print("FX status/null mismatches:", int(fx.value.isna().ne(fx.observation_status.eq("missing")).sum()))

,period,primary_value_usd,net_weight_kg,quantity,quantity_unit,unit_value_usd_per_kg
79,201608,2.381065e+07,1350.89,0.0,N/A,17625.898138
137,202107,8.887831e+08,0.00,0.0,N/A,770449.716747
192,202602,1.536218e+08,NaN,0.0,N/A,191086.142792


Maximum unit-value reconciliation error: 0.0
FX missing values: 474 of 12183 (3.89%)
FX status/null mismatches: 0


### Provenance metadata
The notebook records sources embedded in the files. Live-source verification and release calendars are outside this audit.

In [6]:
for path in files:
    name = path.stem.split("__", 1)[1]
    frame = frames[name]
    metadata = pq.read_metadata(path).metadata or {}
    envelopes = json.loads(metadata.get(b"scrape_harness.record_envelopes.v1", b"[]"))
    sources = set(frame.source_url.dropna()) if "source_url" in frame else set()
    raw_paths = set()
    envelope_fields = set()
    for envelope in envelopes:
        envelope_fields.update(envelope)
        if envelope.get("raw_artifact_path"):
            raw_paths.add(envelope["raw_artifact_path"])
        source = envelope.get("target_metadata", {}).get("source_url")
        if source:
            sources.add(source)
    print(name)
    print("  Source URLs:", sorted(sources) or "Not embedded")
    print("  Envelope fields:", sorted(envelope_fields))
    print("  Referenced raw artifacts:", len(raw_paths),
          "present in repository:", sum((root / p).exists() for p in raw_paths))

chinadata_ev_production
  Source URLs: ['https://chinadata.live/api/v2/data/china-ev-production']
  Envelope fields: ['identity', 'raw_artifact_path', 'source_id', 'target_metadata']
  Referenced raw artifacts: 1 present in repository: 0
pgm_price_history
  Source URLs: ['https://app-prod-pricechart-001-e7cqh7hkdba2c6gs.westeurope-01.azurewebsites.net/price-history?charttype=line&lang=en&dateFrom=2018-01-01&dateTo=2026-09-10&price=USD&metals=pt,pd,rh,ru,ir']
  Envelope fields: ['raw_artifact_path', 'source_id', 'target_metadata']
  Referenced raw artifacts: 1 present in repository: 0
chinadata_vehicle_production
  Source URLs: ['https://chinadata.live/api/v2/data/china-vehicle-production']
  Envelope fields: ['identity', 'observed_at', 'raw_artifact_path', 'source_id', 'target_metadata']
  Referenced raw artifacts: 1 present in repository: 0
fred_zafprmito01ixobm_history
  Source URLs: ['https://fred.stlouisfed.org/series/ZAFPRMITO01IXOBM']
  Envelope fields: ['identity', 'raw_artifact

un_comtrade_sa_refined_rhodium_monthly
  Source URLs: Not embedded
  Envelope fields: ['raw_artifact_path', 'source_id', 'target_metadata']
  Referenced raw artifacts: 196 present in repository: 0
fred_dexsfus_history
  Source URLs: ['https://fred.stlouisfed.org/series/DEXSFUS']
  Envelope fields: ['identity', 'raw_artifact_path', 'source_id', 'target_metadata']
  Referenced raw artifacts: 1 present in repository: 0


## Added attachments

These checks cover the four imported Parquet tables and renamed CFTC CSV. Source-preserving import logic is in `scripts/import_trial_attachments.py`. The import checks every merged observation's value on round-trip, matching series metadata at the 2002/2003 boundary, and annual supply/demand identities. Source workbooks remain available under `data/sources/`.

### Read and validate the additional research tables

In [7]:
additional_files = {
    "mining": "sa_mining_production_sales_monthly.parquet",
    "rhodium_supply_demand": "jm_rhodium_supply_demand_annual.parquet",
    "russia_risk": "russia_geopolitical_risk_index.parquet",
    "china_ev": "iea_china_ev_sales.parquet",
}
added = {name: pd.read_parquet(root / "data" / filename) for name, filename in additional_files.items()}
added["cftc"] = pd.read_csv(root / "data/cftc_pt_pd_open_interest.csv", dtype={"cftc_contract_market_code": str})
keys = {
    "mining": ["period", "series_id"],
    "rhodium_supply_demand": ["year", "section", "series_name"],
    "russia_risk": ["period"],
    "china_ev": ["region", "category", "parameter", "mode", "powertrain", "year", "unit"],
    "cftc": ["report_date_as_yyyy_mm_dd", "cftc_contract_market_code"],
}
additional_inventory = []
for name, frame in added.items():
    assert not frame.duplicated().any() and not frame.duplicated(keys[name]).any()
    assert not np.isinf(frame.select_dtypes("number")).any().any()
    additional_inventory.append({"dataset": name, "rows": len(frame), "columns": len(frame.columns),
                                 "duplicate_keys": int(frame.duplicated(keys[name]).sum())})
display(pd.DataFrame(additional_inventory))
receipt = json.loads((root / "data/sources/import_manifest.json").read_text())
for source in receipt["sources"]:
    assert hashlib.sha256((root / "data" / source["file"]).read_bytes()).hexdigest() == source["sha256"]
print("All six attached source files match the import manifest hashes.")

,dataset,rows,columns,duplicate_keys
0,mining,23202,16,0
1,rhodium_supply_demand,672,11,0
2,russia_risk,1520,3,0
3,china_ev,197,8,0
4,cftc,2114,6,0


All six attached source files match the import manifest hashes.


### Merged mining coverage and annual rhodium quantities

In [8]:
mining = added["mining"]
mining_coverage = mining.groupby("series_id").agg(first=("period", "min"), last=("period", "max"), months=("value", "size"))
assert mining.value.notna().all()
for _, row in mining_coverage.iterrows():
    assert len(pd.date_range(row["first"], row["last"], freq="MS")) == row["months"]
display(mining_coverage.groupby(["first", "last", "months"]).size().rename("series_count").reset_index())
display(mining_coverage.loc[["FMP23023", "FMS23023", "MVK23023"]])
rhodium = added["rhodium_supply_demand"]
display(rhodium.loc[rhodium.value.isna()].groupby(["section", "series_name"]).agg(first=("year", "min"), last=("year", "max"), missing_values=("year", "size")))
totals = rhodium.pivot(index="year", columns="series_name", values="value")
assert rhodium.groupby(["section", "series_name"]).year.nunique().eq(42).all()
assert rhodium.is_report_year.eq(rhodium.year.eq(2026)).all()
np.testing.assert_allclose(totals["Total combined supply"], totals["Total Supply"] + totals["Total secondary supply"])
np.testing.assert_allclose(totals["Movements in Stocks"], totals["Total combined supply"] - totals["Total Demand"])
print("Annual supply and stock-movement identities reconcile for all 42 years; 2026 remains a report-year outlook.")

,first,last,months,series_count
0,1980-01,2002-12,276,3
1,1980-01,2026-07,559,38
2,2003-01,2026-07,283,4


,first,last,months
series_id,,,
FMP23023,1980-01,2026-07,559
FMS23023,2003-01,2026-07,283
MVK23023,1980-01,2026-07,559


,,first,last,missing_values
section,series_name,,,
primary_supply,Zimbabwe,1985,1999,15


Annual supply and stock-movement identities reconcile for all 42 years; 2026 remains a report-year outlook.


### CFTC calendar, Russia measures, and China EV series

In [9]:
cftc = added["cftc"]
calendar_rows = []
for contract, group in cftc.groupby("cftc_contract_market_code"):
    dates = pd.DatetimeIndex(pd.to_datetime(group.report_date_as_yyyy_mm_dd)).sort_values()
    calendar_rows.append({"contract": contract, "rows": len(group), "first": str(dates.min().date()),
                         "last": str(dates.max().date()), "Monday": int((dates.dayofweek == 0).sum()),
                         "Tuesday": int((dates.dayofweek == 1).sum()), "Wednesday": int((dates.dayofweek == 2).sum()),
                         "max_gap_days": int(pd.Series(dates).diff().dt.days.max())})
assert cftc.notna().all().all()
display(pd.DataFrame(calendar_rows))
risk = added["russia_risk"]
risk_rows = []
for column in ["GPRC_RUS", "GPRHC_RUS"]:
    valid = risk.loc[risk[column].notna()]
    assert len(pd.date_range(valid.period.min(), valid.period.max(), freq="MS")) == len(valid)
    risk_rows.append({"measure": column, "first_non_null": valid.period.min(), "last_non_null": valid.period.max(),
                      "non_null": len(valid), "null": int(risk[column].isna().sum())})
display(pd.DataFrame(risk_rows))
china = added["china_ev"]
assert china.notna().all().all()
china_coverage = china.groupby(["parameter", "powertrain", "unit"]).agg(first=("year", "min"), last=("year", "max"), rows=("value", "size"))
assert china_coverage.rows.eq(china_coverage["last"] - china_coverage["first"] + 1).all()
display(china_coverage)
print("These annual China sales measures remain separate from monthly China production.")

,contract,rows,first,last,Monday,Tuesday,Wednesday,max_gap_days
0,075651,1057,2006-06-13,2026-09-08,13,1043,1,8
1,076651,1057,2006-06-13,2026-09-08,13,1043,1,8


,measure,first_non_null,last_non_null,non_null,null
0,GPRC_RUS,1985-01,2026-08,500,1020
1,GPRHC_RUS,1900-01,2026-08,1520,0


first  \
parameter             powertrain unit                                        
Battery deployment    EV         GWh                                  2015   
EV sales              BEV        Vehicles                             2010   
                      EV         Vehicles                             2010   
                      FCEV       Vehicles                             2021   
                      PHEV       Vehicles                             2010   
EV sales share        EV         percent                              2010   
EV stock              BEV        Vehicles                             2010   
                      EV         Vehicles                             2010   
                      FCEV       Vehicles                             2021   
                      PHEV       Vehicles                             2010   
EV stock share        EV         percent                              2010   
Electricity demand    EV         GWh                                  2010   
Oil displacement Mlge EV         Million litres gasoline equivalent   2010   
Oil displacement, Mbd EV         Million barrels per day              2010   

                                                                     last  \
parameter             powertrain unit                                       
Battery deployment    EV         GWh                                 2025   
EV sales              BEV        Vehicles                            2025   
                      EV         Vehicles                            2025   
                      FCEV       Vehicles                            2025   
                      PHEV       Vehicles                            2025   
EV sales share        EV         percent                             2025   
EV stock              BEV        Vehicles                            2025   
                      EV         Vehicles                            2025   
                      FCEV       Vehicles                            2025   
                      PHEV       Vehicles                            2025   
EV stock share        EV         percent                             2025   
Electricity demand    EV         GWh                                 2025   
Oil displacement Mlge EV         Million litres gasoline equivalent  2025   
Oil displacement, Mbd EV         Million barrels per day             2025   

                                                                     rows  
parameter             powertrain unit                                      
Battery deployment    EV         GWh                                   11  
EV sales              BEV        Vehicles                              16  
                      EV         Vehicles                              16  
                      FCEV       Vehicles                               5  
                      PHEV       Vehicles                              16  
EV sales share        EV         percent                               16  
EV stock              BEV        Vehicles                              16  
                      EV         Vehicles                              16  
                      FCEV       Vehicles                               5  
                      PHEV       Vehicles                              16  
EV stock share        EV         percent                               16  
Electricity demand    EV         GWh                                   16  
Oil displacement Mlge EV         Million litres gasoline equivalent    16  
Oil displacement, Mbd EV         Million barrels per day               16

These annual China sales measures remain separate from monthly China production.


## Coverage notes

- The target has 7,672 quotes from 16 September 1996 through 15 September 2026, with no blank or nonpositive prices and no duplicate dates. Quote gaps longer than five calendar days run from 31 December 2002 to 17 January 2003, 31 December 2015 to 11 January 2016, and 30 December 2016 to 6 January 2017. These intervals have not been interpolated. Other calendar gaps are at most five days.
- China vehicle production omits all of 1996, every January from 1997 onward, and some February observations. January/February reporting conventions need to be considered when interpreting these absences. The monthly new-energy-vehicle series begins in March 2023.
- December 2020 is absent from South African rhodium exports, including the source response retrieved on 15 September 2026. July 2021 and February 2026 unit values have been calculated from the source's alternate kilogram quantities; the original zero/missing net weights remain unchanged.
- The FRED mining series ends in October 2023. The separate Statistics South Africa tables continue through July 2026 with different definitions and a different index base.
- Zimbabwe supply is blank for 1985–1999. Russia's recent risk measure begins in 1985; its earlier blanks are outside that measure's coverage.
- The FX file retains 474 source nulls on its weekday grid. These include non-quotation dates; no synthetic exchange-rate quotes have been inserted.
- Reporting dates and reference periods do not establish when information became available. Historical release timing and revisions should be accounted for in the backtest.


## Johnson Matthey target and restored trade values

In [10]:
target = pd.read_parquet(root / "data/johnson_matthey_rhodium_daily.parquet")
assert len(target) == 7672
assert target.observation_date.is_unique
assert target.notna().all().all()
assert target.observation_date.min() == pd.Timestamp("1996-09-16")
assert target.price_usd_per_troy_oz.gt(0).all()
assert np.allclose(target.price_usd_per_lb, target.price_usd_per_troy_oz * (7000 / 480))
display(target.agg({"observation_date": ["min", "max"]}))
dates = target.observation_date.sort_values()
gaps = pd.DataFrame({"previous_quote": dates.shift(), "next_quote": dates, "calendar_days": dates.diff().dt.days})
display(gaps[gaps.calendar_days.gt(5)])
restored = trade[trade.unit_value_weight_basis.eq("alternate_quantity_kg")]
assert set(restored.period.astype(str)) == {"202107", "202602"}
assert np.allclose(restored.unit_value_usd_per_kg, restored.primary_value_usd / restored.alternate_quantity_kg)
assert trade.unit_value_usd_per_kg.notna().all()
original_trade = pd.read_parquet(root / "data/sources/un_comtrade_original.parquet")
pd.testing.assert_frame_equal(trade[original_trade.columns.drop("unit_value_usd_per_kg")], original_trade.drop(columns="unit_value_usd_per_kg"))
display(restored[["period", "primary_value_usd", "net_weight_kg", "alternate_quantity_kg", "unit_value_usd_per_kg"]])
raw_target = pd.read_csv(root / "data/sources/jm_rhodium_new_york_daily.csv", skiprows=1)
raw_dates = pd.to_datetime(raw_target.Date, format="%d-%b-%Y")
assert target.quote_region.eq("New York").all()
assert target.observation_date.tolist() == raw_dates.tolist()
np.testing.assert_array_equal(target.price_usd_per_troy_oz, raw_target.Rhodium)
provenance = json.loads((root / "data/sources/target_provenance.json").read_text())
assert hashlib.sha256((root / "data/sources/jm_rhodium_new_york_daily.csv").read_bytes()).hexdigest() == provenance["sha256"]


,observation_date
min,1996-09-16
max,2026-09-15


,previous_quote,next_quote,calendar_days
1613,2002-12-31,2003-01-17,17.0
4931,2015-12-31,2016-01-11,11.0
5181,2016-12-30,2017-01-06,7.0


,period,primary_value_usd,net_weight_kg,alternate_quantity_kg,unit_value_usd_per_kg
137,202107,8.887831e+08,0.0,1153.59,770449.716747
192,202602,1.536218e+08,NaN,803.94,191086.142792
